In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
#Model class inheriting nn.module

class Model(nn.Module):
    #input layer -> hidden layer1 -> HL2 -> output
    #chose 8 arribraterly 

    def __init__(self, inputs=4, h1=8, h2=8, outputs=3):
        super().__init__()
        self.fc1 = nn.Linear(inputs,h1)
        self.fc2 = nn.Linear(h1,h2)
        self.out = nn.Linear(h2,outputs)
    
    def forward(self,x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.out(x)

        return x







In [ ]:
torch.manual_seed(41)

model = Model()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
%matplotlib inline

#import iris dataset inot dataframe
url = 'https://gist.githubusercontent.com/curran/a08a1080b88344b0c8a7/raw/0e7a9b0a5d22642a06d3d5b9bcbad9890c8ee534/iris.csv'
my_df = pd.read_csv(url)


In [ ]:
#cleaned up species to calssify with nummbers
my_df['species'] = my_df['species'].replace('setosa',0.0)
my_df['species'] = my_df['species'].replace('versicolor',1.0)
my_df['species'] = my_df['species'].replace('virginica',2.0)


In [ ]:
#train model
x = my_df.drop('species',axis=1)
y = my_df['species']

#convert to arrays
x = x.values
y = y.values


In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
#split data to training and testing
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.2, random_state=42)

In [ ]:
#convert to tensors
x_train = torch.FloatTensor(x_train)
x_test = torch.FloatTensor(x_test)

y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)

In [ ]:
#measure error, to see how far from nn pred

criterion = nn.CrossEntropyLoss()

#pick an optimizer
#using adam optimizer, learning rate (if err doesnt go down lower it)

optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

In [ ]:
#training
#how many epochs

epochs  = 800
losses = [] #to keep track

for i in range(epochs):
    y_pred = model.forward(x_train)

    loss = criterion(y_pred, y_train)

    losses.append(loss.item())

    if i % 10 == 0:
        print(f'Epoch: {i} and loss: {loss}')

    #backprop
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


In [ ]:
plt.plot(range(epochs),losses)
plt.ylabel("loss")
plt.xlabel("epoch")

In [ ]:
#eval model on test 
with torch.no_grad():
    y_eval = model.forward(x_test) 
    loss = criterion(y_eval,y_test)

In [ ]:
loss

correct = 0

with torch.no_grad():
    for i, data in enumerate(x_test):
        y_val = model.forward(data)


        print(f'{i+1} {str(y_val)}\t{y_test[i]}')

        if y_val.argmax().item() == y_test[i]:
            correct += 1

#correct #30/30 correct




In [19]:
#extra test cases not from dataset

test_cases = [
    # Iris pallida — large flowers, similar to virginica
    [7.2, 3.8, 6.8, 2.3],   # expect: virginica

    # Iris sibirica — small, delicate, similar to setosa
    [4.5, 3.9, 1.2, 0.2],   # expect: setosa

    # Iris germanica — medium, similar to versicolor
    [6.1, 2.8, 4.5, 1.4],   # expect: versicolor

    # Deliberately ambiguous — between versicolor and virginica
    [6.5, 3.0, 5.2, 1.9],   # expect: virginica (borderline)

    # Outlier — very large, nothing like training data
    [8.5, 4.5, 7.5, 3.2],   # expect: virginica (closest, but out of distribution)
]



for features in test_cases:
    t = torch.tensor(features, dtype=torch.float32)
    with torch.no_grad():
        out = model(t)
        pred = torch.argmax(out).item()
    labels = {0: 'setosa', 1: 'versicolor', 2: 'virginica'}
    print(f"{features} -> {labels[pred]}")

[7.2, 3.8, 6.8, 2.3] → virginica
[4.5, 3.9, 1.2, 0.2] → setosa
[6.1, 2.8, 4.5, 1.4] → versicolor
[6.5, 3.0, 5.2, 1.9] → virginica
[8.5, 4.5, 7.5, 3.2] → virginica
